In [3]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import pandas as pd

In [4]:
df = pd.read_csv("../data/voice_features/voice_dataset_labeled.csv")

df.head()

,file,user_id,covid_status,pitch,spectral_centroid,zcr,jitter,shimmer,hnr,mfcc_1,...,mfcc_5,mfcc_6,mfcc_7,mfcc_8,mfcc_9,mfcc_10,mfcc_11,mfcc_12,mfcc_13,label
0,/Users/aadiii/Desktop/PulseIQ-AI/data/Coswara-...,9hXEs9OejdVxG6JJGCyKQpqVvy43,positive_moderate,1311.654175,1799.875023,0.031378,0.021356,0.133057,12.828806,-428.88983,...,15.640949,-15.971164,-21.472250,-10.734362,-11.321500,-13.427616,-11.184027,-7.961647,-7.676924,1
1,/Users/aadiii/Desktop/PulseIQ-AI/data/Coswara-...,9hXEs9OejdVxG6JJGCyKQpqVvy43,positive_moderate,1204.279907,1428.154074,0.029119,0.019582,0.109823,15.473836,-423.96936,...,10.317966,-5.761888,-18.675251,-16.562550,-13.115095,-12.825630,-10.813223,-7.029854,-5.865900,1
2,/Users/aadiii/Desktop/PulseIQ-AI/data/Coswara-...,o5T5M7lfD0aUpgHcFprqdICYcgd2,recovered_full,1336.240234,4383.912482,0.077219,0.023354,0.088631,17.858712,-448.26636,...,23.619053,13.418217,-2.999132,1.750096,5.951513,-1.198462,-8.297428,-6.682641,-2.950694,1
3,/Users/aadiii/Desktop/PulseIQ-AI/data/Coswara-...,o5T5M7lfD0aUpgHcFprqdICYcgd2,recovered_full,1504.194214,4021.171949,0.079062,0.019765,0.079095,18.108631,-406.47003,...,26.291855,9.338547,-10.927941,-2.514927,2.074677,-7.480949,-11.926127,-6.714749,-4.547808,1
4,/Users/aadiii/Desktop/PulseIQ-AI/data/Coswara-...,ZPup16g7iWhU8qGhXNlgpgiN7ow2,no_resp_illness_exposed,1649.551025,2166.407122,0.069087,0.016123,0.112984,15.364097,-454.14822,...,16.839968,-7.684074,-8.459986,-21.481264,-12.294534,3.653709,-5.710431,-2.271479,-1.403067,1


In [5]:
# Separate features and label

X = df.drop(columns=["file", "user_id", "covid_status", "label"])
y = df["label"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (5411, 19)
y shape: (5411,)


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (4328, 19)
Test shape: (1083, 19)


In [7]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"  # IMPORTANT for imbalance
)

model.fit(X_train, y_train)

print("Model trained!")

Model trained!


In [8]:
from sklearn.metrics import classification_report, roc_auc_score

# Predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Metrics
print(classification_report(y_test, y_pred))

auc = roc_auc_score(y_test, y_prob)
print("AUROC:", auc)

              precision    recall  f1-score   support

           0       0.70      0.78      0.73       563
           1       0.72      0.63      0.68       520

    accuracy                           0.71      1083
   macro avg       0.71      0.71      0.71      1083
weighted avg       0.71      0.71      0.71      1083

AUROC: 0.7656578767591202


In [9]:
import numpy as np

print("Predicted labels:", np.unique(y_pred, return_counts=True))
print("Actual labels:", np.unique(y_test, return_counts=True))

Predicted labels: (array([0, 1]), array([627, 456]))
Actual labels: (array([0, 1]), array([563, 520]))


In [10]:
# Custom threshold

threshold = 0.3  # lower than default 0.5

y_pred_custom = (y_prob >= threshold).astype(int)

from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_custom))

              precision    recall  f1-score   support

           0       0.84      0.24      0.38       563
           1       0.54      0.95      0.69       520

    accuracy                           0.58      1083
   macro avg       0.69      0.60      0.53      1083
weighted avg       0.70      0.58      0.52      1083



In [11]:
# Check missing values

print(X.isna().sum())

pitch                  0
spectral_centroid      0
zcr                    0
jitter               171
shimmer              173
hnr                  141
mfcc_1                 0
mfcc_2                 0
mfcc_3                 0
mfcc_4                 0
mfcc_5                 0
mfcc_6                 0
mfcc_7                 0
mfcc_8                 0
mfcc_9                 0
mfcc_10                0
mfcc_11                0
mfcc_12                0
mfcc_13                0
dtype: int64


In [12]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="mean")

X_imputed = imputer.fit_transform(X)

# Convert back to DataFrame
X_imputed = pd.DataFrame(X_imputed, columns=X.columns)

print("NaNs after fix:", X_imputed.isna().sum().sum())

NaNs after fix: 0


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [14]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Before:", y_train.value_counts())
print("After:", pd.Series(y_train_resampled).value_counts())

Before: label
0    2250
1    2078
Name: count, dtype: int64
After: label
0    2250
1    2250
Name: count, dtype: int64


In [15]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train_resampled, y_train_resampled)

print("Model retrained with SMOTE!")

Model retrained with SMOTE!


In [16]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))

print("AUROC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.70      0.72      0.71       563
           1       0.69      0.66      0.67       520

    accuracy                           0.69      1083
   macro avg       0.69      0.69      0.69      1083
weighted avg       0.69      0.69      0.69      1083

AUROC: 0.7616409345539008


In [17]:
threshold = 0.3

y_pred_custom = (y_prob >= threshold).astype(int)

print(classification_report(y_test, y_pred_custom))

              precision    recall  f1-score   support

           0       0.85      0.23      0.36       563
           1       0.53      0.95      0.69       520

    accuracy                           0.58      1083
   macro avg       0.69      0.59      0.52      1083
weighted avg       0.70      0.58      0.52      1083



In [18]:
df = pd.read_csv("../data/voice_features/voice_dataset_labeled.csv")

print(df["label"].value_counts())

label
0    2813
1    2598
Name: count, dtype: int64


In [19]:
X = df.drop(columns=["file", "user_id", "covid_status", "label"])
y = df["label"]

In [20]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="mean")
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [22]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

In [23]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_resampled, y_train_resampled)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [24]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("AUROC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.70      0.72      0.71       563
           1       0.69      0.66      0.67       520

    accuracy                           0.69      1083
   macro avg       0.69      0.69      0.69      1083
weighted avg       0.69      0.69      0.69      1083

AUROC: 0.7616409345539008


In [25]:
threshold = 0.25

y_pred_custom = (y_prob >= threshold).astype(int)

print(classification_report(y_test, y_pred_custom))

              precision    recall  f1-score   support

           0       0.86      0.12      0.21       563
           1       0.51      0.98      0.67       520

    accuracy                           0.53      1083
   macro avg       0.68      0.55      0.44      1083
weighted avg       0.69      0.53      0.43      1083



In [26]:
df = pd.read_csv("../data/voice_features/voice_dataset_labeled_full.csv")

df["label"].value_counts()

label
0    2813
1    2598
Name: count, dtype: int64

In [27]:
X = df.drop(columns=["file", "user_id", "covid_status", "label"])
y = df["label"]

In [28]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="mean")
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

In [29]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [30]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,  # slightly stronger
    random_state=42
)

model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [31]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("AUROC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.70      0.75      0.72       563
           1       0.71      0.64      0.67       520

    accuracy                           0.70      1083
   macro avg       0.70      0.70      0.70      1083
weighted avg       0.70      0.70      0.70      1083

AUROC: 0.7686466730427655


In [32]:
import sys
print(sys.executable)

/Users/aadiii/Desktop/PulseIQ-AI/venv/bin/python


In [33]:
import pandas as pd

df = pd.read_csv("../data/voice_features/voice_dataset_labeled_full.csv")
print(df.shape)

(5411, 23)


In [34]:
X = df.drop(columns=["file", "user_id", "covid_status", "label"])
y = df["label"]

print(X.shape, y.shape)

(5411, 19) (5411,)


In [35]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="mean")
X_imputed = imputer.fit_transform(X)

In [36]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [37]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [38]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("AUROC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.69      0.75      0.72       563
           1       0.70      0.63      0.67       520

    accuracy                           0.70      1083
   macro avg       0.70      0.69      0.69      1083
weighted avg       0.70      0.70      0.69      1083

AUROC: 0.7708942478480667


In [39]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

model.fit(X_train, y_train)

/Users/aadiii/Desktop/PulseIQ-AI/venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [23:31:56] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [40]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

from sklearn.metrics import classification_report, roc_auc_score

print(classification_report(y_test, y_pred))
print("AUROC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.70      0.74      0.72       563
           1       0.70      0.65      0.67       520

    accuracy                           0.70      1083
   macro avg       0.70      0.70      0.70      1083
weighted avg       0.70      0.70      0.70      1083

AUROC: 0.7735500068315344


In [41]:
import joblib
import os

os.makedirs("../models", exist_ok=True)
joblib.dump(model, "../models/respiratory_model.pkl")
joblib.dump(imputer, "../models/respiratory_imputer.pkl")
print("✅ Respiratory model saved!")

✅ Respiratory model saved!


In [42]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_acc = cross_val_score(model, X_imputed, y, cv=skf, scoring="accuracy")
cv_auc = cross_val_score(model, X_imputed, y, cv=skf, scoring="roc_auc")

print("Respiratory Model - 5-Fold Cross Validation:")
print(f"  Accuracy : {cv_acc.mean():.4f} +/- {cv_acc.std():.4f}")
print(f"  AUROC    : {cv_auc.mean():.4f} +/- {cv_auc.std():.4f}")

/Users/aadiii/Desktop/PulseIQ-AI/venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [23:31:57] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/aadiii/Desktop/PulseIQ-AI/venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [23:31:57] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/aadiii/Desktop/PulseIQ-AI/venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [23:31:58] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/aadiii/Desktop/PulseIQ-AI/venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [23:31:58] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.

Respiratory Model - 5-Fold Cross Validation:
  Accuracy : 0.6808 +/- 0.0108
  AUROC    : 0.7483 +/- 0.0126
